# Laboratorio 8

In [23]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import random
import pyreadr
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_val_predict
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 9)
plt.style.use('ggplot')

In [10]:
result = pyreadr.read_r('listings.RData')
df = result[list(result.keys())[0]]

### 1. Conjuntos Train y Test

In [13]:
#Limpieza de datos

df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")

df = df.dropna(subset=["price"])

In [14]:
def categorize_price(price):
    if price <= 120:
        return "Economico"
    elif price <= 326:
        return "Intermedio"
    else:
        return "Caro"

df["price_category"] = df["price"].apply(categorize_price)

/var/folders/xv/g38xvw9j28vc78v68ljk5g900000gn/T/ipykernel_12048/3701250134.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price_category"] = df["price"].apply(categorize_price)


In [15]:
df['price_category'].value_counts()

price_category
Intermedio    37882
Economico     19310
Caro          19054
Name: count, dtype: int64

In [20]:
# variables predictoras
X = df.drop(columns=["price", "price_category"])

# variable objetivo (clasificación)
y = df["price_category"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y   # MUY IMPORTANTE
)

In [25]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")

y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_test = pd.read_csv("data/y_test.csv").values.ravel()

### 2. Exploración y Transformación

In [26]:
pd.DataFrame(X_train).isnull().sum().sort_values(ascending=False).head(10)

review_scores_checkin          9391
review_scores_location         9391
review_scores_value            9391
review_scores_communication    9391
review_scores_accuracy         9391
review_scores_cleanliness      9391
review_scores_rating           9388
reviews_per_month              9388
bathrooms                        11
maximum_nights_avg_ntm            0
dtype: int64

In [27]:
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [29]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [30]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

print(pd.Series(y_train).value_counts())

(53372, 15)
(22874, 15)
135.0     357
100.0     343
90.0      340
110.0     332
150.0     325
         ... 
2854.0      1
2991.0      1
3034.0      1
2718.0      1
4455.0      1
Name: count, Length: 2076, dtype: int64


### 3. Variable Respuesta

Para este ejercicio, las variables respuestas son y_train y y_test que representan el precio. Estas representan:
 - Económico
 - Intermedio
- Caro

### 4. Modelos SVM

In [31]:
# tomar 20% de los datos

sample_idx = np.random.choice(len(X_train_scaled), size=int(len(X_train_scaled)*0.2), replace=False)

X_train_sample = X_train_scaled[sample_idx]
y_train_sample = y_train[sample_idx]

In [ ]:
resultados = []

# MODELO 1: LINEAL
svm_linear = SVC(kernel='linear', C=1)
svm_linear.fit(X_train_sample, y_train_sample)

y_pred_lin = svm_linear.predict(X_test_scaled)
acc_lin = accuracy_score(y_test, y_pred_lin)

resultados.append(("Linear", acc_lin))


# MODELO 2: RBF
svm_rbf = SVC(kernel='rbf', C=1, gamma='scale')
svm_rbf.fit(X_train_sample, y_train_sample)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
acc_rbf = accuracy_score(y_test, y_pred_rbf)

resultados.append(("RBF", acc_rbf))


# MODELO 3: POLINOMIAL
svm_poly = SVC(kernel='poly', degree=3, C=1)
svm_poly.fit(X_train_sample, y_train_sample)

y_pred_poly = svm_poly.predict(X_test_scaled)
acc_poly = accuracy_score(y_test, y_pred_poly)

resultados.append(("Polynomial", acc_poly))


# RESULTADOS
for modelo, acc in resultados:
    print(f"{modelo}: {acc:.4f}")